# The verifier is an executable task definition

Day 23 · Chapter 13 · CPU worked notebook

Predict: should a response containing the correct number anywhere receive credit? Should a truncated answer? Read Chapter13 section13.6 before revealing the reference.

Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Make the prediction above before inspecting the preview.

![Saved reference 1: verifier contract](../figures/chapter-13/day-23-02_verifier_contract-01.png)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


In [ ]:
from dongxi_llms.grpo_lab import verify_integer, verify_tokens, NUMBER_START, EOS
cases = ["35", " +35 ", "35 0", "answer35", "35.0", "٣٥", "0"]
strict = [verify_integer(text, 35) for text in cases]
substring = ["35" in text for text in cases]
fig, ax = plt.subplots(figsize=(9, 3.8))
positions = torch.arange(len(cases)).numpy()
ax.barh(positions-.16, substring, height=.3, label="Broken substring rule")
ax.barh(positions+.16, strict, height=.3, label="Strict one-integer rule")
ax.set(yticks=positions, yticklabels=cases, xlabel="Accepted (1) or rejected (0)")
ax.legend()
plt.show()

Reference explanation: substring credit accepts malformed competing answers. Strict parsing makes formatting an explicit task constraint; it may reject a mathematically valid explanation outside that constraint.

Exercise: add one true positive, one ambiguous answer and one Unicode edge case. Require EOS for token verification.

In [ ]:
# Reference solution: deliberately explicit contract, no arbitrary code execution.
assert verify_integer("-35", -35)
assert not verify_integer("35 or -35", 35)
assert not verify_integer("３５", 35)
assert verify_tokens([NUMBER_START+3, EOS, 0], 3) == 1.
assert verify_tokens([NUMBER_START+3], 3) == 0.
print("Parser/EOS cases pass.")
print("Task coverage remains final-integer correctness; reasoning validity is unverified.")

Controlled change: permit a declared trailing period, then describe how the verifier version and tests must change. Evidence boundary: tested strings establish the narrow parser contract, not an adversarial proof over all outputs.